# FREQ-NESS Interactive Pipeline

This notebook is the interactive, single-condition companion to
`FREQNESS_MainPipeline.py`. It follows the same analysis order while allowing
intermediate variables and figures to be inspected directly in Jupyter or VS
Code.

For unattended processing of every condition in `FREQNESS_Data`, use
`FREQNESS_MainPipeline.py`. The implementation under `src/freqness` is the
reusable package engine and should not be removed.

**Data format:** `FREQNESS_Startup` automatically loads `.mat` files from
`FREQNESS_Data`; manual conversion to NumPy arrays is not required.

## 1. Imports

Install the package from the `python` directory before running the notebook:

```text
python -m pip install -e ".[visualize]"
```

In [ ]:
import warnings
from pathlib import Path

import numpy as np

from freqness import (
    FREQNESS_BackProjection,
    FREQNESS_CompGradients,
    FREQNESS_CrossCoupling,
    FREQNESS_EntropyLandscape,
    FREQNESS_ExponentialDK,
    FREQNESS_FreqGradients,
    FREQNESS_NetworkEstimation,
    FREQNESS_NetworkRemoval,
    FREQNESS_Startup,
    FREQNESS_Visualizer,
    FREQNESSPipelineConfig,
)

## 2. Configuration

These settings are shared with the batch pipeline. Adjust the sampling rate,
frequency vector, component selections, and display options for your data.

In [ ]:
config = FREQNESSPipelineConfig(
    toolbox_root=None,
    srate=250.0,
    plot_all=True,
    show=True,
    save_nifti=False,
)

print(f"Frequencies to analyze: {config.frex}")

## 3. Load data and MNI coordinates

Each subfolder inside `FREQNESS_Data` represents one condition or group. Each
participant is stored as one `.mat` file containing a variables-by-time
matrix.

In [ ]:
all_data, MNI, root = FREQNESS_Startup(config.toolbox_root)
condition_directories = sorted(
    path for path in (root / "FREQNESS_Data").iterdir() if path.is_dir()
)
condition_names = [path.name for path in condition_directories]

if len(condition_names) != len(all_data):
    raise RuntimeError("Dataset folders changed while data were loading.")

print(f"Found {len(all_data)} condition(s): {condition_names}")

## 4. Select one condition

Change `CONDITION_INDEX` to inspect another condition. Python uses zero-based
indices, so `0` selects the first folder, `1` the second, and so forth.

In [ ]:
CONDITION_INDEX = 0

if not all_data:
    raise RuntimeError(
        "No data were loaded. Add condition folders and participant .mat files "
        "inside FREQNESS_Data."
    )
if not 0 <= CONDITION_INDEX < len(all_data):
    raise IndexError("CONDITION_INDEX is outside the available condition range.")

data = all_data[CONDITION_INDEX]
condition_name = condition_names[CONDITION_INDEX]
if data is None:
    raise RuntimeError(
        f"Condition {condition_name!r} contains no participant .mat files."
    )

print(f"Selected condition: {condition_name}")
print(f"Data shape: {np.shape(data)}")

## 5. FREQNESS_NetworkEstimation

This core prerequisite estimates the frequency-resolved networks and produces
the `FREQ` structure used by the downstream analyses.

In [ ]:
FREQ = FREQNESS_NetworkEstimation(
    data,
    np.asarray(config.frex, dtype=float),
    config.srate,
    **dict(config.network_options),
)

FREQ

## 6. FREQNESS_Visualizer

The pattern visualization requires MNI coordinates matching the variables in
the input data. Set `save_nifti=True` in the configuration to export NIfTI
images.

In [ ]:
if MNI is None:
    warnings.warn(
        "No valid MNI coordinates were loaded; skipping FREQNESS_Visualizer."
    )
    visualization = None
else:
    landscape_frex = (
        np.asarray(config.frex, dtype=float)
        if config.landscape_frex is None
        else np.asarray(config.landscape_frex, dtype=float)
    )
    output_directory = (
        root
        if config.visualizer_output_directory is None
        else Path(config.visualizer_output_directory).expanduser().resolve()
    )
    visualization = FREQNESS_Visualizer(
        FREQ,
        {
            "frex": landscape_frex,
            "ncomps": config.landscape_ncomps,
        },
        {
            "frex": np.asarray(config.pattern_frex, dtype=float),
            "ncomps": config.pattern_ncomps,
            "path_output": output_directory,
            "MNI_coords": MNI,
        },
        plot_all=config.plot_all,
        save_nifti=config.save_nifti,
        show=config.show,
    )

## 7. FREQNESS_EntropyLandscape

In [ ]:
H2, ED = FREQNESS_EntropyLandscape(
    FREQ,
    plot_all=config.plot_all,
    show=config.show,
)

H2

## 8. FREQNESS_ExponentialDK

In [ ]:
decayCoeff, expGoodFit = FREQNESS_ExponentialDK(
    FREQ,
    which_comp=config.expdk_which_comp,
    range2fit=config.expdk_range2fit,
    plot_all=config.plot_all,
    show=config.show,
)

decayCoeff

## 9. FREQNESS_FreqGradients and FREQNESS_CompGradients

Both analyses require MNI coordinates matching the spatial patterns.

In [ ]:
if MNI is None:
    warnings.warn(
        "No valid MNI coordinates were loaded; skipping FREQNESS_FreqGradients."
    )
    freqGradCoeff = None
    freqGoodFit = None
else:
    freqGradCoeff, freqGoodFit = FREQNESS_FreqGradients(
        FREQ,
        MNI,
        frex2model=config.freqgrad_frex2model,
        comp2model=config.freqgrad_comp2model,
        plot_all=config.plot_all,
        show=config.show,
    )

In [ ]:
if MNI is None:
    warnings.warn(
        "No valid MNI coordinates were loaded; skipping FREQNESS_CompGradients."
    )
    compGradCoeff = None
    compGoodFit = None
else:
    compGradCoeff, compGoodFit = FREQNESS_CompGradients(
        FREQ,
        MNI,
        freq2model=config.compgrad_freq2model,
        comps2model=config.compgrad_comps2model,
        plot_all=config.plot_all,
        show=config.show,
    )

## 10. FREQNESS_CrossCoupling

In [ ]:
CFC = FREQNESS_CrossCoupling(
    FREQ,
    config.lfo_freq,
    MNI=MNI,
    plot_all=config.plot_all,
    show=config.show,
)

CFC

## 11. FREQNESS_BackProjection

In [ ]:
backProj = FREQNESS_BackProjection(
    FREQ,
    config.backproj_freq2project,
    comps2project=config.backproj_comps2project,
)

print(f"Backprojected data shape: {backProj.shape}")

## 12. FREQNESS_NetworkRemoval

The cleaned-network re-estimation remains in the pipeline rather than inside
`FREQNESS_NetworkRemoval`, matching the MATLAB and Python batch workflows.

In [ ]:
analyzed_data = np.asarray(data, dtype=float)[:, : FREQ.ts.shape[1], ...]
dataClean, removedActivity = FREQNESS_NetworkRemoval(
    FREQ,
    analyzed_data,
    config.netrem_freq2remove,
    comps2remove=config.netrem_comps2remove,
)

FREQ_clean = None
clean_visualization = None
if config.netrem_plot_landscape:
    FREQ_clean = FREQNESS_NetworkEstimation(
        dataClean,
        np.asarray(config.frex, dtype=float),
        config.srate,
        **dict(config.network_options),
    )
    clean_visualization = FREQNESS_Visualizer(
        FREQ_clean,
        {
            "frex": np.asarray(config.frex, dtype=float),
            "ncomps": config.netrem_landscape_ncomps,
        },
        None,
        save_nifti=False,
        show=config.show,
    )

print(f"Cleaned data shape: {dataClean.shape}")

## 13. Collected outputs

The variables remain available individually. This dictionary provides one
convenient summary for the selected condition.

In [ ]:
outputs = {
    "condition": condition_name,
    "FREQNESS_NetworkEstimation": FREQ,
    "FREQNESS_Visualizer": visualization,
    "FREQNESS_EntropyLandscape": {"H2": H2, "ED": ED},
    "FREQNESS_ExponentialDK": {
        "decayCoeff": decayCoeff,
        "goodFit": expGoodFit,
    },
    "FREQNESS_FreqGradients": {
        "gradCoeff": freqGradCoeff,
        "goodFit": freqGoodFit,
    },
    "FREQNESS_CompGradients": {
        "gradCoeff": compGradCoeff,
        "goodFit": compGoodFit,
    },
    "FREQNESS_CrossCoupling": CFC,
    "FREQNESS_BackProjection": backProj,
    "FREQNESS_NetworkRemoval": {
        "dataClean": dataClean,
        "removedActivity": removedActivity,
        "FREQ_clean": FREQ_clean,
        "visualization": clean_visualization,
    },
}

list(outputs)

## Event-related extension: FREQNESS_InducedResponses

Induced responses are intentionally not part of the main pipeline because they
require experiment-specific event samples, epoch windows, and baseline
windows. Call `FREQNESS_InducedResponses` separately when those inputs are
available; see `README.md` for an example.